# GIGABYTE AORUS MASTER 16 AM6H — RAG Demo

**開始前：Runtime → Change runtime type → T4 GPU**

---

```
【第一次使用】
  Section A：安裝環境  ← 跑完後必須 Restart Runtime
  Section B：重啟後初始化
  Section C ~ G：正常執行

【之後重連 Colab（runtime 未斷線）】
  只跑 Section B，然後從 C 繼續

【之後重連 Colab（runtime 已斷線）】
  重跑 Section A → Restart → Section B → Section C 起
```

---
## Section A：安裝環境（只需執行一次）
### ⚠️ 跑完 A4 後必須 Restart Runtime，再從 Section B 繼續

In [ ]:
# A1：確認 GPU
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv
!nvcc --version | grep release

In [ ]:
# A2：Clone repo
import os

REPO_URL = "https://github.com/chiahua-yang/GIGABYTE-RAG-test.git"
BRANCH   = "claude/build-rag-product-qa-2nMZM"
REPO_DIR = "/content/GIGABYTE-RAG-test"

if os.path.isdir(REPO_DIR):
    !git -C {REPO_DIR} pull origin {BRANCH}
else:
    !git clone -b {BRANCH} {REPO_URL} {REPO_DIR}

os.chdir(REPO_DIR)
print("Working dir:", os.getcwd())

In [ ]:
# A3：安裝 Python 套件（不含 llama-cpp-python）
!pip install requests beautifulsoup4 lxml sentence-transformers numpy huggingface-hub -q
print("Done")

In [ ]:
# A4：安裝 llama-cpp-python（預編譯 CUDA wheel，< 2 分鐘）
# cu124 wheel 在 CUDA 12.8 環境向下相容
!pip install llama-cpp-python \
  --extra-index-url https://abetlen.github.io/llama-cpp-python/whl/cu124 \
  --force-reinstall --no-cache-dir
print("\nInstall complete — numpy/tensorflow warnings above are harmless.")
print(">>> 現在執行 Runtime → Restart session，再從 Section B 繼續 <<<

---
## Section B：重啟後初始化（每次 runtime 重啟都要跑）

In [ ]:
# B1：設定工作目錄與 sys.path
import os, sys

REPO_DIR = "/content/GIGABYTE-RAG-test"
os.chdir(REPO_DIR)
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

print("Working dir:", os.getcwd())

In [ ]:
# B2：修正 CUDA symlink（解決 libcudart.so.12 找不到的問題）
import glob, subprocess

# 找出系統上實際的 libcudart 檔案
candidates = sorted(glob.glob('/usr/local/cuda*/lib64/libcudart.so*'))
print("Found CUDA libs:", candidates)

if candidates:
    src = candidates[-1]  # 取最新版
    subprocess.run(['ln', '-sf', src, '/usr/lib/x86_64-linux-gnu/libcudart.so.12'], check=True)
    subprocess.run(['ldconfig'], capture_output=True)
    print(f"Linked: {src} → libcudart.so.12 ✓")
else:
    print("WARNING: No libcudart found — will try CPU-only fallback")

In [ ]:
# B3：確認所有套件正常
import requests, json, pickle
import numpy as np
from llama_cpp import Llama
from sentence_transformers import SentenceTransformer

print("numpy:", np.__version__)
print("All imports OK ✓")

---
## Section C：爬取規格資料

### ⚠️ GIGABYTE 會封鎖 Colab IP（403），必須手動上傳 HTML
1. 用自己的瀏覽器開啟：`https://www.gigabyte.com/tw/Laptop/AORUS-MASTER-16-AM6H/sp`
2. 等規格表完全顯示（滾動確認看到 RTX、DDR5）
3. `Ctrl+S`（Mac: `Cmd+S`）→ 存檔類型選**「網頁，僅 HTML」**
4. 執行 C1 上傳

In [ ]:
# C1：上傳本地 HTML 檔案
# 如果已上傳過（data/spec_page.html 存在），跳過此格
import os
if os.path.exists('data/spec_page.html'):
    print('data/spec_page.html 已存在，跳過上傳')
else:
    from google.colab import files
    import shutil
    os.makedirs('data', exist_ok=True)
    print('請選擇儲存的 .html 檔案 ...')
    uploaded = files.upload()
    if uploaded:
        fname = list(uploaded.keys())[0]
        shutil.move(fname, 'data/spec_page.html')
        print(f'✓ 已儲存為 data/spec_page.html ({os.path.getsize("data/spec_page.html"):,} bytes)')

In [ ]:
# C2：確認 HTML 關鍵字（應全部 ✓）
with open('data/spec_page.html', encoding='utf-8', errors='replace') as f:
    html = f.read()
print(f'File size: {len(html):,} chars')
for kw in ['RTX', 'DDR5', 'Intel', '顯示', 'BZH', 'BYH', 'BXH', 'tabSpec']:
    print(f"  '{kw}': {'✓' if kw in html else '✗ 頁面可能未完整載入就儲存'}")

In [ ]:
# C3：解析規格資料 → data/specs.json
!mkdir -p data
!python -m src.data.scraper

In [ ]:
# C4：確認解析結果
import json
with open('data/specs.json') as f:
    specs = json.load(f)
print(f'Total records: {len(specs)}')
print(f'Models: {sorted({r["model"] for r in specs})}')
print('\n前 3 筆:')
for r in specs[:3]:
    print(f'  {r}')

---
## Section D：建立向量索引
Embedding model：`BAAI/bge-small-zh-v1.5`（~93MB，CPU）  
Retrieval：Hybrid BM25 + embedding cosine similarity

In [ ]:
# D1：語意分組 chunking
!python -m src.data.chunker

In [ ]:
# D2：建立向量索引（自動下載 bge-small-zh-v1.5）
# 在 kernel 內執行確保 BM25.__module__ = 'src.rag.indexer'（而非 __main__）
from src.rag.indexer import main as build_index_main
build_index_main()

In [ ]:
# D3：確認 chunks
with open('data/chunks.json') as f:
    chunks = json.load(f)
print(f'Total chunks: {len(chunks)}')
for c in chunks:
    print(f"  [{c['model']:35s}] {c['category']}")

---
## Section E：下載 LLM 與載入元件

**Qwen2.5-3B-Instruct-Q4_K_M**（~2.0 GB）  
VRAM 預算：LLM ~2.0GB + KV Cache ~0.8GB ≈ 3.0GB（< 4GB ✓）

In [ ]:
# E1：下載 LLM（約 2GB，需要幾分鐘）
import os
os.makedirs('models', exist_ok=True)

from src.rag.generator import download_model
path = download_model()
print('Downloaded:', path)

In [ ]:
# E2：載入所有元件
from sentence_transformers import SentenceTransformer
from src.rag.indexer import load_index, retrieve, EMBED_MODEL
from src.rag.generator import load_model, build_prompt, stream_generate

print('Loading index ...')
index = load_index()

print('Loading embedding model ...')
embed_model = SentenceTransformer(EMBED_MODEL, device='cpu')

print('Loading LLM (GPU offload) ...')
llm = load_model(n_gpu_layers=-1)

print('\n✓ All components loaded!')

In [ ]:
# E3：確認 VRAM 使用量（應 < 4GB）
!nvidia-smi --query-gpu=name,memory.used,memory.total --format=csv

---
## Section F：RAG 問答測試

In [ ]:
# F1：定義 ask() 函數
def ask(question: str, top_k: int = 3):
    print(f'Q: {question}')
    print('-' * 60)

    chunks = retrieve(question, index, embed_model, top_k=top_k)
    print('[Retrieved chunks]')
    for c in chunks:
        print(f"  {c['id']}  (score={c['score']:.4f})")
    print()

    messages = build_prompt(question, chunks)
    print('[Answer]')
    answer, metrics = '', {}
    for token, m in stream_generate(llm, messages):
        print(token, end='', flush=True)
        answer += token
        if m:
            metrics = m

    print(f"\n\n[Metrics] TTFT={metrics.get('ttft')}s | "
          f"TPS={metrics.get('tps')} tok/s | "
          f"tokens={metrics.get('total_tokens')}")
    return answer, metrics

In [ ]:
# F2：測試 1 — 直接查找（CPU）
ask('這台筆電的 CPU 是什麼型號？')

In [ ]:
# F3：測試 2 — 型號指定（GPU）
ask('AM6H-BZH 的顯卡是什麼？VRAM 有多少？')

In [ ]:
# F4：測試 3 — 跨型號比較
ask('三個型號 BZH、BYH、BXH 的 GPU 差異是什麼？')

In [ ]:
# F5：測試 4 — 英文提問
ask('What GPU does the AM6H-BYH have and how much VRAM?')

---
## Section G：Benchmark 評測（20 題）

In [ ]:
# G1：執行完整 benchmark
!python -m src.evaluation.benchmark --top-k 3 --max-tokens 256 --output data/benchmark_results.json

In [ ]:
# G2：顯示結果摘要
import json
with open('data/benchmark_results.json') as f:
    results = json.load(f)

print('=' * 50)
print('BENCHMARK SUMMARY')
print('=' * 50)
for k, v in results['summary'].items():
    print(f'  {k}: {v}')

print()
print('PER-QUESTION RESULTS')
print('-' * 50)
for r in results['results']:
    ok = '✓' if r['retrieval_recall'] else '✗'
    print(f"  [{r['id']:6s}] {ok} | kw={r['keyword_hit_rate']:.0%} | "
          f"ttft={r['metrics'].get('ttft','?')}s | tps={r['metrics'].get('tps','?')}")